## Why Tokenisation Matter

- First bottleneck in NLP Pipeline
- IF , tokenizer fragements common words into too many pieces, your model wastes capacity reassembling them. (we need to avoid this!)
- Contrarily, if it groups too many words as a single token, it can't handle rare or unseen words. 
- BPE should be engineered so LLMS can hadle common and rare words
- Every decision in tokenisation step results in model quality, training speed and vocab_size


### Three tokenisation stategies 
- **Character Level** : 
    - vocab_size = ~ 256 (ASCII) or ~ 65k (Unicode)
    - Handles Out Of Vocabulary (OOV) since character are limited always 
    - Sequence length is very long (5-7 x word count), assuming mean character per word is 5-7. 
    - Semantic density is low (each token = 1 character) 

- **Word Level** : 
    - Vocab_size = 100k + 
    - Cannot handle OOV words, uses '\<UNK\>' token when it encounters unknown token 
    - Sequence length is short (1x word count) 
    - Semantic density is high (each token  = 1 word)

- **Subword (BPE)** : 
    - Vocab_size = 10k-100k (tunable to dataset) 
    - Handles OOV words since it decomposes it into subwords
    - Sequence length is mordern (between characterl & word level)
    - Semantic density is medium (common words stay whole)

> BPE is character level tokenization that has been iteratively compressed by merging frequent pairs. It starts with combining characters, then combining words , 
stopping when vocab_size is attained

ALgorithm in plan English 
```text 
START with a vocabulary = set of all individual characters in the corpus
REPEAT num_merges times:
    1. Look at the corpus (split into characters/current tokens)
    2. COUNT every adjacent pair of tokens
    3. FIND the most frequent pair
    4. MERGE that pair into a single new token
    5. ADD the new token to the vocabulary
    6. REPLACE all occurrences of that pair in the corpus
END

```

In [3]:
# minimal BPE tokenizer 
# libraries used: stdlibs, collections(for counting), matplollib (viz)

corpus = [
    "low low low low low",
    "lowest lowest",
    "newer newer newer",
    "wider wider",
    "new new",
    "the cat sat on the mat",
    "the dog sat on the log",
    "the cat and the dog",
    "this is the lowest of the low",
    "a newer and wider perspective",
]

print(f"Corpus: {len(corpus)} sentences")
print(f"Total characters: {sum(len(s) for s in corpus)}")

Corpus: 10 sentences
Total characters: 188


This corpus is designed to have :
- Repeated words (low x6, the x7) -- so BPE has strong frequency signals 
- SHared subwords (low in lowes, new in newer, er in newer/wider) so you can see BPE discover morphemes 
- Enough diversity to show real tokenization behavior

**Senior-dev insight:** In production, BPE is trained on gigabyte of text (BooksCorpus for GPT - 1 = ~800 M words). The algorithm is same - only the scale changes

### Character level baseline
STRATEGY 1 : Treat every character as a token.

Pros: 
    - Zero OOV - any Unicode String can be tokenized 
    - Tiny vocab (just the characters that appear) 

Cons: 
    - Sequences become VERY long (every char = 1 token) 
    - Model must learn to assemble characters into words -> wastes capacity
    - Positional Embeddings (512 max in GPT-1) cover much less text

Engineering tradeoff: We're trading VOCAB_SIZE for SEQUENCE_LENGTH. 
The model's context window is fixed at 512 tokens. At char-level, 
that's roughly 512 characters ~ 80 -100 words. At word level, that is 512 words ~ a full page. THis directly hurts the model.

In [5]:
def char_tokenize(text:str) -> list[str]: 
    """Split text into individual characters""" 
    return list(text) 


sample = "I am anvesh dange"
char_tokens = char_tokenize(sample); print(f"{char_tokens=}")
char_vocab = sorted(set(char_tokenize(" ".join(corpus)))) 

print(f"Sample: {sample}") 
print(f"Char Tokens: {char_tokens}")
print(f"Num Tokens: {len(char_tokens)}")
print(f"Char Vocab Size: {len(char_vocab)}")
print(f"Char Vocab : {char_vocab}")

char_tokens=['I', ' ', 'a', 'm', ' ', 'a', 'n', 'v', 'e', 's', 'h', ' ', 'd', 'a', 'n', 'g', 'e']
Sample: I am anvesh dange
Char Tokens: ['I', ' ', 'a', 'm', ' ', 'a', 'n', 'v', 'e', 's', 'h', ' ', 'd', 'a', 'n', 'g', 'e']
Num Tokens: 17
Char Vocab Size: 19
Char Vocab : [' ', 'a', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'l', 'm', 'n', 'o', 'p', 'r', 's', 't', 'v', 'w']


In [9]:
set(char_tokenize(" ".join(corpus)))

{' ',
 'a',
 'c',
 'd',
 'e',
 'f',
 'g',
 'h',
 'i',
 'l',
 'm',
 'n',
 'o',
 'p',
 'r',
 's',
 't',
 'v',
 'w'}